In [34]:
# Base functions

import requests, re
import time
from bs4 import BeautifulSoup
from urllib.parse import urlparse

LANG_PATTERN = re.compile(r"^[a-z]{2}-[A-Z]{2}$")

def clean_path_segments(path):
    parts = path.strip("/").split("/")
    return [p for p in parts if not LANG_PATTERN.match(p)]

HEADERS = {
    "accept": "application/json",
    "content-type": "application/json",
    "user-agent": "Mozilla/5.0"
}


class WorkdayScraper:

    FACETS = {
        "locationCountry": ["c4f78be1a8f14da0ab49ce1162348a5e"],
        "locationRegionStateProvince": ["701eb5584934425d930bc84b9e8b04eb"]
    }

    def __init__(self, base_url, search_text=None, list_url=None, detail_url=None,facets=FACETS , limit=20):

        self.base_url = base_url
        # self.list_url = list_url
        # self.detail_url = detail_url
        self.limit = limit

        self.payload = {
            "limit": limit,
            "offset": 0
        }

        if search_text:
            self.payload["searchText"] = search_text

        if facets:
            self.payload["appliedFacets"] = facets

        parsed = urlparse(base_url)

        # example: cat.wd5.myworkdayjobs.com
        host = parsed.netloc
        
        # tenant = first part before dot
        tenant = host.split('.')[0]
        
        # example path: /CaterpillarCareers/login
        path_parts = parsed.path.strip('/').split('/')
        
        if len(path_parts) < 1:
            raise ValueError("Invalid Workday login URL")
            
        # Remove language segments like en-US
        path_parts = clean_path_segments(parsed.path)
    
        if len(path_parts) == 0:
            raise ValueError("Invalid Workday URL")
        
        org = path_parts[0]

        base = f"https://{host}"

        self.base_url = base
        self.detail_url = f"{base}/wday/cxs/{tenant}/{org}"
        self.list_url = f"{self.detail_url}/jobs"

        # print(self.base_url)
        # print(self.detail_url)
        # print(self.list_url)

    def is_india_location(self, loc_text):
        # print(loc_text)
        if not loc_text:
            return False
    
        loc_text = loc_text.lower()
    
        # obvious match
        if "india" in loc_text:
            return True
    
        # fallback: common Indian cities (extend as needed)
        indian_cities = [
            "bangalore", "bengaluru", "hyderabad", "pune",
            "mumbai", "delhi", "noida", "gurgaon", "chennai",
            "kolkata", "ahmedabad"
        ]
    
        return any(city in loc_text for city in indian_cities)

    def is_within_24h(self,posted_text):
    
        if not posted_text:
            return False
    
        posted_text = posted_text.lower()
    
        if "today" in posted_text:
            return True
    
        if "yesterday" in posted_text:
            return True
    
        m = re.search(r'(\d+)', posted_text)
        if m:
            days = int(m.group(1))
            return days <= 1
    
        return False
    
    
    def fetch_jobs(self):
    
        jobs = []
        seen = set()
    
        while True:
    
            r = requests.post(self.list_url, headers=HEADERS, json=self.payload)
    
            if r.status_code != 200:
                print(r.text)
                print("Request failed:", r.status_code)
                break
    
            data = r.json()
            postings = data.get("jobPostings", [])
    
            if not postings:
                break
    
            new_jobs = 0
    
            for job in postings:
    
                key = job.get("externalPath")
                posted = job.get("postedOn")
                locText = job.get("locationsText")

                if not ( self.is_within_24h(posted) and self.is_india_location(locText) ):
                    continue
    
                if key and key not in seen:
                    seen.add(key)
                    jobs.append(job)
                    new_jobs += 1
    
            print(f"Offset {self.payload['offset']} → {new_jobs} new jobs")
    
            self.payload["offset"] += self.limit
    
            if len(postings) < self.limit:
                break
    
            time.sleep(0.4)
    
        return jobs


    def fetch_job_detail(self, external_path):

        url = f"{self.detail_url}/{external_path}"

        r = requests.get(url, headers=HEADERS)

        if r.status_code != 200:
            print("Failed detail fetch:", external_path)
            return None

        return r.json()


    def clean_html(self, html):

        if not html:
            return ""

        soup = BeautifulSoup(html, "html.parser")
        return soup.get_text(separator="\n").strip()


    def print_job(self, detail):

        info = detail.get("jobPostingInfo", {})

        print("\n===================================")
        print("Title:", info.get("title"))
        print("Job ID:", info.get("jobReqId"))
        print("Location:", info.get("location"))
        print("Posted:", info.get("postedOn"))

        desc = info.get("jobDescription")
        description = self.clean_html(desc)

        print(description)
        print("===================================\n")


    def run(self):

        jobs = self.fetch_jobs()

        print("\nTotal unique jobs found:", len(jobs))

        for job in jobs:

            path = job.get("externalPath")

            if not path:
                continue

            detail = self.fetch_job_detail(path)

            if detail:
                self.print_job(detail)

            time.sleep(0.3)

In [ ]:
BASE = "https://walmart.wd5.myworkdayjobs.com/WalmartExternal"

scraper = WorkdayScraper(
    base_url=BASE,
    search_text="Senior Software Engineer"
)

scraper.run()

In [38]:
BASE = "https://netflix.wd1.myworkdayjobs.com"

LIST_URL = BASE + "/wday/cxs/netflix/Netflix/jobs"
DETAIL_URL = BASE + "/wday/cxs/netflix/Netflix"

scraper = WorkdayScraper(
    base_url=BASE,
    list_url=LIST_URL,
    detail_url=DETAIL_URL,
    search_text="Senior Software Engineer",
    facets=None
)

scraper.run()

{"errorCode":"HTTP_500","errorCaseId":"6EA276MMUSDMPZ","httpStatus":500,"message":"","messageParams":{}}
Request failed: 500

Total unique jobs found: 0


In [32]:
BASE = "https://travelhrportal.wd1.myworkdayjobs.com/Jobs"

scraper = WorkdayScraper(
    base_url=BASE,
    facets=None,
    search_text="Senior Software Engineer"
)

scraper.run()

Bangalore, India
Offset 0 → 1 new jobs

Total unique jobs found: 1

Title: Technical Lead
Job ID: J-80069
Location: Bangalore, India
Posted: Posted Today
Amex GBT is a place where colleagues find inspiration in travel as a force for good and – through their work – can make an impact on our industry. We’re here to help our colleagues achieve success and offer an inclusive and collaborative culture where your voice is valued.
What You’ll do 
Lead and mentor a team of backend/Boomi Integration engineers in delivering high-quality software solutions 
Define technical architecture and design patterns for the GBT  mid,  back office and Boomi Integration systems 
Drive technical decision-making and establish development guidelines 
Conduct code reviews and ensure consistency to coding standards and quality guidelines 
Collaborate with product managers, architects, and customers on technical roadmap planning
Design and develop complex Dell Boomi Integrations, backend services and APIs using Ja

In [29]:
BASE = "https://boeing.wd1.myworkdayjobs.com/EXTERNAL_CAREERS"

# LIST_URL = BASE + "/wday/cxs/boeing/EXTERNAL_CAREERS/jobs"
# DETAIL_URL = BASE + "/wday/cxs/boeing/EXTERNAL_CAREERS"

scraper = WorkdayScraper(
    base_url=BASE,
    # list_url=LIST_URL,
    # detail_url=DETAIL_URL,
    search_text="Senior Software Engineer",
    facets=None,
)

scraper.run()

Offset 0 → 0 new jobs
USA - Berkeley, MO
Offset 20 → 0 new jobs
USA - Oklahoma City, OK
IND - Bangalore, India
Offset 40 → 1 new jobs
USA - Herndon, VA
USA - Puyallup, WA
USA - Berkeley, MO
IND - Bangalore, India
USA - Colorado Springs, CO
USA - Berkeley, MO
Offset 60 → 1 new jobs
USA - Hazelwood, MO
2 Locations
USA - Long Beach, CA
Offset 80 → 0 new jobs
2 Locations
Offset 100 → 0 new jobs
USA - Berkeley, MO
USA - El Segundo, CA
5 Locations
Offset 120 → 0 new jobs
IND - Bangalore, India
USA - Mesa, AZ
USA - Berkeley, MO
USA - North Charleston, SC
USA - Hazelwood, MO
Offset 140 → 1 new jobs
USA - Berkeley, MO
USA - Fort Walton Beach, FL
BRA - Sao Jose dos Campos, Brazil
USA - Tukwila, WA
Offset 160 → 0 new jobs
6 Locations
USA - El Segundo, CA
Offset 180 → 0 new jobs

Total unique jobs found: 3

Title: Experienced Full Stack Software Developer
Job ID: JR2026500583
Location: IND - Bangalore, India
Posted: Posted Yesterday
Experienced Full Stack Software Developer
Company:
Boeing India P

In [28]:
BASE = "https://cat.wd5.myworkdayjobs.com/CaterpillarCareers/login"

# LIST_URL = BASE + "/wday/cxs/cat/CaterpillarCareers/jobs"
# DETAIL_URL = BASE + "/wday/cxs/cat/CaterpillarCareers"

scraper = WorkdayScraper(
    base_url=BASE,
    search_text="Senior Software Engineer",
    facets=None,
)

scraper.run()

Chennai, Tamil Nadu
2 Locations
2 Locations
2 Locations
2 Locations
Bangalore, Karnataka
Offset 0 → 2 new jobs
Peterborough, United Kingdom
2 Locations
Laval, Quebec
Bangalore, Karnataka
Bangalore, Karnataka
Offset 20 → 2 new jobs
2 Locations
2 Locations
Offset 40 → 0 new jobs
Pontiac, Illinois
Montreal, Quebec
Offset 60 → 0 new jobs
Offset 80 → 0 new jobs

Total unique jobs found: 4

Title: Senior Software Engineer
Job ID: R0000346691
Location: Chennai, Tamil Nadu
Posted: Posted Yesterday
Career Area:
Technology, Digital and Data
Job Description:
Your Work Shapes the World at Caterpillar Inc. 
When you join Caterpillar, you're joining a global team who cares not just about the work we do – but also about each other.  We are the makers, problem solvers, and future world builders who are creating stronger, more sustainable communities. We don't just talk about progress and innovation here – we make it happen, with our customers, where we work and live. Together, we are building a better

In [27]:
BASE = "https://toyota.wd503.myworkdayjobs.com/TMNA"

# LIST_URL = BASE + "/wday/cxs/cat/CaterpillarCareers/jobs"
# DETAIL_URL = BASE + "/wday/cxs/cat/CaterpillarCareers"

scraper = WorkdayScraper(
    base_url=BASE,
    search_text="Senior Software Engineer",
    facets=None,
)

scraper.run()

Plano, Texas
Georgetown, Kentucky
Princeton, Indiana
Offset 0 → 1 new jobs
Offset 20 → 0 new jobs

Total unique jobs found: 1

Title: Lead Seibi Technician - Production Engineering
Job ID: 10322762
Location: Princeton, Indiana
Posted: Posted Today
Overview
Who we are
Collaborative. Respectful. A place to dream and do. These are just a few words that describe what life is like at Toyota. As one of the world’s most admired brands, Toyota is growing and leading the future of mobility through innovative, high-quality solutions designed to enhance lives and delight those we serve. We’re looking for talented team members who want to Dream. Do. Grow. with us.
To save time applying, Toyota does not offer sponsorship of job applicants for employment-based visas or any other work authorization for this position at this time.
Who are we looking for?
Toyota’s Body Production Engineering (BPE) Department is looking for a passionate and highly motivated Lead Technician.
The primary responsibility of